# Step 2 - Cleaning the Géo'DAE dataset

**Goal:** turn the raw file into a clean table that the Streamlit app can load quickly.

What this notebook does:
1. downloads the raw file if it is not already on the computer,
2. keeps only the useful columns and gives them clear English names,
3. converts dates and yes/no columns,
4. finds the department of each defibrillator,
5. checks the location, the equipment (battery and electrodes) and the opening hours,
6. adds one column per step of our story (the "funnel"),
7. saves the result as `data/clean/geodae_clean.parquet`.

We **do not delete** problematic rows: we add flag columns instead.
This way, the app can show both the data quality problems and the clean results.

Source: https://www.data.gouv.fr/datasets/geodae-base-nationale-des-defibrillateurs (Licence Ouverte 2.0)

> If the last cell fails with an error about `pyarrow`, run `pip install pyarrow` in Anaconda Prompt and restart the notebook.

In [1]:
   pip install pyarrow

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



## 1. Imports and settings

In [2]:
import os
import urllib.request
import pandas as pd

# Where the raw file comes from (data.gouv.fr)
DATA_URL = "https://www.data.gouv.fr/api/1/datasets/r/edb6a9e1-2f16-4bbf-99e7-c3eb6b90794c"

# Where files are stored in the project
RAW_PATH = "data/raw/geodae.csv"
CLEAN_PATH = "data/clean/geodae_clean.parquet"

# Fixed reference date used to decide if a battery or electrodes are expired.
# It is fixed so that the results are the same every time the notebook is run.
REFERENCE_DATE = pd.Timestamp("2026-10-17")

C:\Users\Solene_NDS\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 2. Download the raw file (only if needed)

The project instructions ask for a download script instead of putting the big file on GitHub.
If the file is already in `data/raw/`, nothing is downloaded.

In [3]:
# Download the raw file only if it is not already on the computer
if not os.path.exists(RAW_PATH):
    os.makedirs("data/raw", exist_ok=True)
    print("Downloading the file (about 75 MB), please wait...")
    urllib.request.urlretrieve(DATA_URL, RAW_PATH)

print("Raw file ready:", RAW_PATH)

Raw file ready: data/raw/geodae.csv


## 3. Load only the useful columns

The raw file has 46 columns. We keep the 19 that our story needs.
Postal codes and INSEE codes are read **as text**, so that a code like `01000` keeps its first zero.

In [4]:
# Find the separator (comma or semicolon) from the first line of the file
with open(RAW_PATH, encoding="utf-8") as f:
    first_line = f.readline()
separator = ";" if first_line.count(";") > first_line.count(",") else ","

# Columns we keep for the story
COLUMNS = ["gid", "c_nom", "c_lat_coor1", "c_long_coor1",
           "c_com_insee", "c_com_cp", "c_com_nom",
           "c_etat_valid", "c_etat", "c_doublon", "c_dae_mobile",
           "c_etat_fonct", "c_acc", "c_acc_lib",
           "c_dtpr_bat", "c_dtpr_lcad", "c_dermnt", "c_date_instal",
           "c_dispo_horaires"]

# Load the file with only these columns
df = pd.read_csv(RAW_PATH, sep=separator, usecols=COLUMNS,
                 dtype={"c_com_insee": str, "c_com_cp": str},
                 encoding="utf-8", low_memory=False)

df.shape

(189536, 19)

## 4. Give the columns clear English names

In [5]:
df = df.rename(columns={
    "gid": "id",
    "c_nom": "name",
    "c_lat_coor1": "lat",
    "c_long_coor1": "lon",
    "c_com_insee": "insee_code",
    "c_com_cp": "postal_code",
    "c_com_nom": "city",
    "c_etat_valid": "validation_status",
    "c_etat": "status",
    "c_doublon": "is_duplicate",
    "c_dae_mobile": "is_mobile",
    "c_etat_fonct": "working_state",
    "c_acc": "access_type",
    "c_acc_lib": "free_access",
    "c_dtpr_bat": "battery_expiry",
    "c_dtpr_lcad": "electrodes_expiry",
    "c_dermnt": "last_maintenance",
    "c_date_instal": "install_date",
    "c_dispo_horaires": "opening_hours",
})

df.columns.tolist()

['id',
 'validation_status',
 'name',
 'lat',
 'lon',
 'postal_code',
 'insee_code',
 'city',
 'access_type',
 'free_access',
 'install_date',
 'working_state',
 'electrodes_expiry',
 'battery_expiry',
 'last_maintenance',
 'status',
 'is_duplicate',
 'is_mobile',
 'opening_hours']

## 5. Convert yes/no columns and dates

- Yes/no columns are stored as `t` (true) and `f` (false). We turn them into real `True` / `False` values.
- Date columns are stored as text. We turn them into real dates.
  A date that cannot be read (for example a typo) becomes empty (`NaT`).

In [6]:
# Yes/no columns: "t" becomes True, everything else becomes False
for col in ["is_duplicate", "is_mobile", "free_access"]:
    df[col] = df[col] == "t"

# Date columns: text becomes a real date (unreadable values become empty)
for col in ["battery_expiry", "electrodes_expiry", "last_maintenance", "install_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.dtypes

id                            int64
validation_status            object
name                         object
lat                         float64
lon                         float64
postal_code                  object
insee_code                   object
city                         object
access_type                  object
free_access                    bool
install_date         datetime64[ns]
working_state                object
electrodes_expiry    datetime64[ns]
battery_expiry       datetime64[ns]
last_maintenance     datetime64[ns]
status                       object
is_duplicate                   bool
is_mobile                      bool
opening_hours                object
dtype: object

## 6. Find the department of each defibrillator

We need the department to draw the map and to compare with the population.

- We take it from the **INSEE code** of the city (first 2 characters, or 3 for overseas departments like `971`).
- The INSEE code is missing for 16% of the rows, so in that case we use the **postal code** instead.
- Corsica is a special case: INSEE codes start with `2A` / `2B`, but postal codes start with `20`.
  For postal codes we use an approximation: below `20200` = `2A`, otherwise `2B`.

In [7]:
def get_department(insee, postal):
    """Return the department code of a city, or None if it cannot be found."""
    # 1. Choose which code to use: INSEE first, postal code if INSEE is missing
    if isinstance(insee, str) and len(insee.strip()) in (4, 5):
        code = insee.strip().zfill(5)      # zfill adds a lost leading zero
    elif isinstance(postal, str) and postal.strip().isdigit() and len(postal.strip()) in (4, 5):
        code = postal.strip().zfill(5)
    else:
        return None

    # 2. Overseas departments use 3 digits (971, 972, 973, 974, 976)
    if code.startswith("97"):
        return code[:3]

    # 3. Corsica written with a postal code (20xxx)
    if code.startswith("20") and code.isdigit():
        return "2A" if int(code) < 20200 else "2B"

    # 4. All other departments: first 2 characters
    return code[:2]


# Apply the function to every row
df["department"] = [get_department(i, p) for i, p in zip(df["insee_code"], df["postal_code"])]

# How many rows still have no department?
print("Rows without department:", df["department"].isna().sum())

Rows without department: 2796


## 7. Check the location

A location is considered **valid** when:
- latitude and longitude are not empty and not equal to 0,
- and the point is inside a box around metropolitan France, **or** the defibrillator is in an overseas department.

The box is approximate (latitude 41 to 51.5, longitude -5.5 to 10). This is an assumption we document in the app.

In [8]:
# Point inside a box around metropolitan France
in_metro_box = df["lat"].between(41.0, 51.5) & df["lon"].between(-5.5, 10.0)

# Defibrillator in an overseas department (971, 972, ...)
is_overseas = df["department"].fillna("").str.startswith("97")

# Coordinates are present and not equal to 0
has_coords = df["lat"].notna() & df["lon"].notna() & (df["lat"] != 0) & (df["lon"] != 0)

df["valid_location"] = has_coords & (in_metro_box | is_overseas)

df["valid_location"].value_counts()

valid_location
True     187636
False      1900
Name: count, dtype: int64

## 8. Check the equipment (battery and electrodes)

Each defibrillator can have an expiry date for its **battery** and its **adult electrodes**.
We compare them with the reference date and give one of three statuses:

| Status | Meaning |
|---|---|
| `Expired` | at least one known date is before the reference date |
| `Valid` | both dates are known and still in the future |
| `Not provided` | not enough information to decide |

In [9]:
def equipment_status(battery_date, electrodes_date):
    """Return 'Expired', 'Valid' or 'Not provided' for one defibrillator."""
    # Keep only the dates that are filled in
    known_dates = [d for d in [battery_date, electrodes_date] if pd.notna(d)]

    # One expired date is enough to say the device is not ready
    for d in known_dates:
        if d < REFERENCE_DATE:
            return "Expired"

    # Both dates known and none expired
    if len(known_dates) == 2:
        return "Valid"

    return "Not provided"


df["equipment_status"] = [equipment_status(b, e)
                          for b, e in zip(df["battery_expiry"], df["electrodes_expiry"])]

df["equipment_status"].value_counts()

equipment_status
Not provided    108661
Valid            54499
Expired          26376
Name: count, dtype: int64

## 9. Read the opening hours

The column `opening_hours` uses the OpenStreetMap format, for example:
- `24/7` = always open,
- `Mo-Fr 09:00-12:00,14:00-18:00` = Monday to Friday, morning and afternoon,
- `Mo-Su off` = closed every day.

**Important assumption:** `Mo-Su off` appears for about 134,000 defibrillators (71%),
even though almost all of them are declared "En fonctionnement".
It is very likely the **default value** of the declaration form, so we treat it as **"Not provided"**, not as "closed".
We do the same for `OFF` and `Mo-Su off; PH off`.

We write a small reader that understands the common patterns of this dataset.
Public holidays (`PH`) are ignored. Texts that the reader cannot understand are marked `Unreadable`.

In [10]:
# Day codes used in the opening-hours text (Monday = 0 ... Sunday = 6)
DAYS = ["Mo", "Tu", "We", "Th", "Fr", "Sa", "Su"]


def parse_opening_hours(text):
    """
    Read an opening-hours text and return:
    - "always" for 24/7,
    - a dictionary {day number: list of (start minute, end minute)},
    - None if the text cannot be read.
    """
    if not isinstance(text, str):
        return None
    text = text.strip()
    if text == "24/7":
        return "always"

    # Start with every day closed
    schedule = {day: [] for day in range(7)}

    # Rules are separated by ";"  (e.g. "Mo-Fr 09:00-18:00; Sa-Su off")
    for rule in text.split(";"):
        rule = rule.strip()
        if rule == "" or rule.startswith("PH"):
            continue  # skip empty parts and public holidays

        # A rule = days + a space + times  (e.g. "Mo-Fr" + "09:00-18:00")
        parts = rule.split(" ", 1)
        if len(parts) != 2:
            return None
        days_text, times_text = parts

        # Find the days of the rule (e.g. "Mo-Fr" or "Sa,Su")
        days = []
        for chunk in days_text.split(","):
            if "-" in chunk:
                first, last = chunk.split("-", 1)
                if first not in DAYS or last not in DAYS:
                    return None
                day = DAYS.index(first)
                while True:
                    days.append(day)
                    if day == DAYS.index(last):
                        break
                    day = (day + 1) % 7
            elif chunk in DAYS:
                days.append(DAYS.index(chunk))
            else:
                return None

        # "off" means closed on these days
        if times_text.strip().lower() == "off":
            for day in days:
                schedule[day] = []
            continue

        # Read the time slots (e.g. "09:00-12:00,14:00-18:00")
        slots = []
        for slot in times_text.split(","):
            try:
                start, end = slot.strip().split("-")
                start_h, start_m = start.split(":")
                end_h, end_m = end.split(":")
                slots.append((int(start_h) * 60 + int(start_m),
                              int(end_h) * 60 + int(end_m)))
            except ValueError:
                return None
        for day in days:
            schedule[day] = slots

    return schedule


# Quick test on a few examples
for example in ["24/7", "Mo-Fr 09:00-12:00,14:00-18:00", "Mo-Fr 09:00-18:00; Sa-Su off", "hello"]:
    print(example, "->", parse_opening_hours(example))

24/7 -> always
Mo-Fr 09:00-12:00,14:00-18:00 -> {0: [(540, 720), (840, 1080)], 1: [(540, 720), (840, 1080)], 2: [(540, 720), (840, 1080)], 3: [(540, 720), (840, 1080)], 4: [(540, 720), (840, 1080)], 5: [], 6: []}
Mo-Fr 09:00-18:00; Sa-Su off -> {0: [(540, 1080)], 1: [(540, 1080)], 2: [(540, 1080)], 3: [(540, 1080)], 4: [(540, 1080)], 5: [], 6: []}
hello -> None


Now we give each defibrillator an **hours category**:

| Category | Meaning |
|---|---|
| `24/7` | open all day, every day |
| `Limited hours` | open only on some days or at some hours |
| `Not provided` | default value or empty |
| `Unreadable` | text that our reader cannot understand |

To be fast, we read each **different** text only once, then copy the result to all rows.

In [11]:
# Values that we consider as "not filled in"
PLACEHOLDERS = ["Mo-Su off", "OFF", "Mo-Su off; PH off"]


def hours_category(text):
    """Return the hours category of one opening-hours text."""
    if not isinstance(text, str) or text.strip() in PLACEHOLDERS:
        return "Not provided"

    schedule = parse_opening_hours(text)
    if schedule is None:
        return "Unreadable"
    if schedule == "always":
        return "24/7"

    # Open all day every day (e.g. "Mo-Su 00:00-23:59") also counts as 24/7
    open_all_day = [any(start <= 0 and end >= 1439 for start, end in schedule[day])
                    for day in range(7)]
    if all(open_all_day):
        return "24/7"

    return "Limited hours"


# Read each different text once, then map the result to all rows
categories = {text: hours_category(text) for text in df["opening_hours"].unique()}
df["hours_category"] = df["opening_hours"].map(categories)

df["hours_category"].value_counts()

hours_category
Not provided     136662
24/7              37004
Limited hours     15637
Unreadable          233
Name: count, dtype: int64

## 10. The funnel: one column per step of the story

Each step keeps the defibrillators of the previous step that also pass a new check:

| Step | Check |
|---|---|
| 1. Usable record | validated, not a duplicate, not "to delete", not a mobile device |
| 2. Valid location | coordinates inside France |
| 3. Declared working | `working_state` = "En fonctionnement" |
| 4. Equipment not expired | battery and electrodes not known to be expired |
| 5. Available 24/7 | opening hours = 24/7 |

Mobile devices (in ambulances, sports clubs...) are removed at step 1 because they have no fixed place.

In [12]:
# Step 1: a usable record in the database
df["step1_record"] = ((df["validation_status"] == "validées")
                      & ~df["is_duplicate"]
                      & (df["status"] != "A supprimer")
                      & ~df["is_mobile"])

# Step 2: + valid location
df["step2_location"] = df["step1_record"] & df["valid_location"]

# Step 3: + declared working
df["step3_working"] = df["step2_location"] & (df["working_state"] == "En fonctionnement")

# Step 4: + equipment not known to be expired
df["step4_equipment"] = df["step3_working"] & (df["equipment_status"] != "Expired")

# Step 5: + available 24/7
df["step5_24_7"] = df["step4_equipment"] & (df["hours_category"] == "24/7")

# Summary table of the funnel
funnel = pd.DataFrame({
    "step": ["0. All declared", "1. Usable record", "2. Valid location",
             "3. Declared working", "4. Equipment not expired", "5. Available 24/7"],
    "defibrillators": [len(df), df["step1_record"].sum(), df["step2_location"].sum(),
                       df["step3_working"].sum(), df["step4_equipment"].sum(),
                       df["step5_24_7"].sum()],
})
funnel["% of all"] = (funnel["defibrillators"] / len(df) * 100).round(1)
funnel

,step,defibrillators,% of all
0,0. All declared,189536,100.0
1,1. Usable record,165658,87.4
2,2. Valid location,163977,86.5
3,3. Declared working,163019,86.0
4,4. Equipment not expired,139898,73.8
5,5. Available 24/7,28191,14.9


## 11. Save the clean file

We save the table in **Parquet** format: it is much smaller and faster to load than CSV.
This is the file the Streamlit app will use. It is small enough to be put on GitHub.

In [13]:
# Create the folder if needed, then save
os.makedirs("data/clean", exist_ok=True)
df.to_parquet(CLEAN_PATH, index=False)

# Compare the file sizes (in MB)
print("Raw CSV size:     ", round(os.path.getsize(RAW_PATH) / 1e6, 1), "MB")
print("Clean Parquet size:", round(os.path.getsize(CLEAN_PATH) / 1e6, 1), "MB")

Raw CSV size:      78.5 MB
Clean Parquet size: 9.9 MB


## Next step

Add the population of each department, so that we can compute the number of defibrillators per 10,000 inhabitants (step 3 of the roadmap).